# Sprint 2: Composite Equity Indicators — Miami-Dade County (Tract-Level)
## AI for Equitable Public Transportation | Deloitte Capstone

This notebook constructs **7 composite equity indicators** and a **final Equity Priority Score** at the **census tract level** by cross-referencing three datasets:

1. **ACS 2023 5-Year** (707 census tracts) — economics, housing, demographics
2. **GeoPackage AllAccess** (36,507 census blocks → aggregated to tracts) — job accessibility by mode and time
3. **GTFS Transit Schedule** — 129 routes, 6,530 stops, 943K stop-time records → aggregated to tracts

**Why tract-level?**
- ACS demographics exist only at tract level — block-level composites repeat the same ACS values across all blocks in a tract, creating an illusion of granularity
- Tracts are the standard planning unit for policy recommendations and federal funding formulas
- All inputs are at their native resolution: no artificial inflation of row count
- Block-level GeoPackage and GTFS data are aggregated UP, preserving real within-tract variation as summary statistics

**Key principle:** Each indicator overlays multiple datasets to answer a question no single dataset can answer alone. The final composite uses multiplicative aggregation (Need × Access Deficit) to concentrate priority where both conditions exist.

**Notebook version:** v3 (March 2026) — tract-level refactor from v2 block-level approach.


## 1. Setup and Configuration

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Libraries loaded successfully.")


### Configuration Cell
All tunable parameters are defined here. Adjust thresholds, weights, and assumptions without touching logic cells below.


In [ ]:
CONFIG = {
    # Time thresholds
    'JOB_ACCESS_TIME': 30,              # minutes — standard for "reasonable commute"
    'WALK_TO_JOB_THRESHOLD': 20,        # minutes — reasonable direct walk to workplace
    'WALK_TO_TRANSIT_THRESHOLD': 10,    # minutes — max acceptable walk to transit stop

    # Walk proxy: no walk GeoPackage available.
    # Using bike_lts1 at 10min as proxy for 20-min walking distance.
    # Bike ~10mph for 10min ≈ 1.7mi; walk ~3mph for 20min ≈ 1mi.
    # This OVERESTIMATES walk access slightly — conservative for deficit scoring.
    'WALK_PROXY_LAYER': 'bi_10_minutes',
    'WALK_PROXY_FILE': '12197701_bi_2021_1200_lts1.gpkg',

    # Mode weights for Transit Dependency and Multimodal indicators
    'TRANSIT_WEIGHT': 0.5,
    'BIKE_WEIGHT': 0.3,
    'WALK_WEIGHT': 0.2,

    # Temporal Service Mismatch
    'PEAK_HOURS': (7, 9),               # 7:00-8:59 AM
    'MIDDAY_HOURS': (10, 14),            # 10:00 AM-1:59 PM
    'EVENING_HOURS': (18, 21),           # 6:00-8:59 PM
    'LATE_HOURS': (21, 24),              # 9:00-11:59 PM
    'POVERTY_THRESHOLD': 13.1,           # county median poverty rate
    'LOW_INCOME_BUCKET_THRESHOLD': 20,   # % of HH earning <$25k
    'SHIFT_WORKER_STATE_RATE': 0.29,     # FL state-level ACS reference

    # Service Coverage weights
    'SVC_WEIGHTS': {
        'stops': 0.25,
        'routes': 0.25,
        'trips': 0.35,
        'wheelchair': 0.15
    },

    # Economic Vulnerability weights
    'ECON_WEIGHTS': {
        'poverty': 0.30,
        'snap': 0.25,
        'rent_burden': 0.25,
        'unemployment': 0.20
    },

    # Composite tier percentile cutoffs
    'TIER_PERCENTILES': [90, 70, 40],    # Critical, High, Moderate, Low
    'TIER_LABELS': ['Critical', 'High', 'Moderate', 'Low'],

    # Time Tax: viable employment threshold
    'VIABLE_JOB_THRESHOLD': 10000,
}

# File paths (relative to this notebook's location in Sprint 2 folder)
PATHS = {
    'acs_clean': 'ACS Tract Level Data/Census_MiamiDade_Tracts_Combined_Clean.csv',
    'gpkg_dir': '../MiamiDadeMpoAllAccessGpkg-expanded',
    'auto_gpkg': '../MiamiDadeMpoAllAccessGpkg-expanded/12197701_au_2021_08.gpkg',
    'transit_gpkg': '../MiamiDadeMpoAllAccessGpkg-expanded/12197701_tr_2021_0700-0859-avg.gpkg',
    'bike_lts1_gpkg': '../MiamiDadeMpoAllAccessGpkg-expanded/12197701_bi_2021_1200_lts1.gpkg',
    'gtfs': '../Sprint 1 EDAs/transit_data.xlsx',
    'tract_crosswalk': 'tab20_tract20_tract10_st12.txt',
}

print("Configuration loaded.")
print(f"  Job access threshold: {CONFIG['JOB_ACCESS_TIME']} min")
print(f"  Walk-to-job threshold: {CONFIG['WALK_TO_JOB_THRESHOLD']} min")
print(f"  Walk-to-transit threshold: {CONFIG['WALK_TO_TRANSIT_THRESHOLD']} min")
print(f"  Walk proxy: bike LTS1 at 10 min")
print(f"  Viable job threshold: {CONFIG['VIABLE_JOB_THRESHOLD']:,}")


### Utility Functions

In [ ]:
def normalize_01(series):
    """Min-max normalize a series to [0, 1]. NaN-safe."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.0, index=series.index)
    return (series - mn) / (mx - mn)

def invert_normalize(series):
    """Normalize then invert: high raw value → low score (good access → low deficit)."""
    normed = normalize_01(series)
    return 1.0 - normed

print("Utility functions defined.")


## 2. Data Loading

### 2.1 ACS Census Tract Data
707 tracts with economics (DP03), housing (DP04), demographics (DP05).
Sentinel value -666666666 already cleaned in prior EDA step.
This is our native tract-level data — no aggregation needed.


In [ ]:
acs = pd.read_csv(PATHS['acs_clean'])
acs['GEOID'] = acs['GEOID'].astype(str).str.zfill(11)

print(f"ACS data loaded: {acs.shape[0]} tracts, {acs.shape[1]} columns")
print(f"GEOID sample: {acs['GEOID'].iloc[0]} (should be 11 digits)")
print(f"\nKey columns available:")
for col in ['poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
            'rent_burden_50pct_plus', 'unemployment_rate_pct',
            'hh_income_under_10k_pct', 'hh_income_10k_15k_pct', 'hh_income_15k_25k_pct']:
    if col in acs.columns:
        print(f"  {col}: median={acs[col].median():.1f}, mean={acs[col].mean():.1f}")
    else:
        print(f"  {col}: NOT FOUND")


### 2.2 GeoPackage Accessibility Data
Loading block-level accessibility and preparing for tract-level aggregation.

**Layer structure:**
- `blocks` layer: geometry (MultiPolygon) + blockid (15-digit FIPS → first 11 = parent tract)
- Time layers (no geometry): `id` = blockid, `w_c000_19` = total jobs, etc.

We load: auto@30min, transit@5-60min (all thresholds for Time Tax), bike_lts1@30min, bike_lts1@10min (walk proxy).


In [ ]:
# Load block geometries (needed for block→tract mapping and GTFS spatial join)
blocks_gdf = gpd.read_file(PATHS['auto_gpkg'], layer='blocks')
blocks_gdf = blocks_gdf.rename(columns={'blockid': 'block_id'})
blocks_gdf['block_id'] = blocks_gdf['block_id'].astype(str)
blocks_gdf['tract_geoid'] = blocks_gdf['block_id'].str[:11]
print(f"Block geometries loaded: {len(blocks_gdf)} blocks, CRS={blocks_gdf.crs}")
print(f"Unique tracts from blocks: {blocks_gdf['tract_geoid'].nunique()}")

# --- Auto 30 min ---
au30 = gpd.read_file(PATHS['auto_gpkg'], layer='au_08_30_minutes')
au30 = au30.rename(columns={'id': 'block_id', 'w_c000_19': 'auto_jobs_30'})
au30['block_id'] = au30['block_id'].astype(str)
au30 = au30[['block_id', 'auto_jobs_30']].copy()
print(f"Auto 30min: {len(au30)} blocks, median jobs={au30['auto_jobs_30'].median():.0f}")

# --- Transit at ALL time thresholds (for Time Tax calculation) ---
transit_thresholds = {}
for t in range(5, 65, 5):
    layer = f'tr_{t}_minutes'
    df = gpd.read_file(PATHS['transit_gpkg'], layer=layer)
    df = df.rename(columns={'id': 'block_id', 'w_c000_19': f'transit_jobs_{t}'})
    df['block_id'] = df['block_id'].astype(str)
    transit_thresholds[t] = df[['block_id', f'transit_jobs_{t}']].copy()

# Merge all transit thresholds into one DataFrame
transit_all = transit_thresholds[5]
for t in range(10, 65, 5):
    transit_all = transit_all.merge(transit_thresholds[t], on='block_id', how='outer')
print(f"Transit all thresholds: {len(transit_all)} blocks")

# --- Bike LTS1 30 min ---
bi30 = gpd.read_file(PATHS['bike_lts1_gpkg'], layer='bi_30_minutes')
bi30 = bi30.rename(columns={'id': 'block_id', 'w_c000_19': 'bike_safe_jobs_30'})
bi30['block_id'] = bi30['block_id'].astype(str)
bi30 = bi30[['block_id', 'bike_safe_jobs_30']].copy()
print(f"Bike LTS1 30min: {len(bi30)} blocks, median jobs={bi30['bike_safe_jobs_30'].median():.0f}")

# --- Walk proxy: Bike LTS1 10 min ---
walk_proxy = gpd.read_file(PATHS['bike_lts1_gpkg'], layer=CONFIG['WALK_PROXY_LAYER'])
walk_proxy = walk_proxy.rename(columns={'id': 'block_id', 'w_c000_19': 'walk_jobs_20'})
walk_proxy['block_id'] = walk_proxy['block_id'].astype(str)
walk_proxy = walk_proxy[['block_id', 'walk_jobs_20']].copy()
print(f"Walk proxy (LTS1 10min): {len(walk_proxy)} blocks, median jobs={walk_proxy['walk_jobs_20'].median():.0f}")

print(f"\n--- GeoPackage loading complete ---")


### 2.3 Block → Tract Aggregation (Accessibility)

**Aggregation strategy:**
- **Mean** for continuous job-accessibility measures (auto/transit/bike/walk jobs reachable)
  - Mean captures average accessibility experience across blocks in a tract
- **Std** as bonus column to capture within-tract variation
- **Block count** per tract to track sample size

**For Time Tax:** We compute time-to-viable at block level FIRST (needs the full transit time series per block), then aggregate the time-tax-minutes to tract level. This preserves the granular transit time curve.


In [ ]:
# Build block-level accessibility DataFrame
block_access = blocks_gdf[['block_id', 'tract_geoid']].copy()
block_access = block_access.merge(au30, on='block_id', how='left')
block_access = block_access.merge(transit_all, on='block_id', how='left')
block_access = block_access.merge(bi30, on='block_id', how='left')
block_access = block_access.merge(walk_proxy, on='block_id', how='left')

# Fill missing accessibility with 0
access_cols = ['auto_jobs_30', 'bike_safe_jobs_30', 'walk_jobs_20'] + [f'transit_jobs_{t}' for t in range(5, 65, 5)]
for col in access_cols:
    if col in block_access.columns:
        block_access[col] = block_access[col].fillna(0)

print(f"Block-level accessibility: {len(block_access)} blocks × {block_access.shape[1]} columns")

# --- Compute Time Tax at block level BEFORE aggregating ---
# (needs full transit time series per block)
transit_cols = [f'transit_jobs_{t}' for t in range(5, 65, 5)]
transit_times = list(range(5, 65, 5))

def compute_time_tax(row):
    """Find minutes for transit to reach viable job threshold."""
    for t, col in zip(transit_times, transit_cols):
        if row[col] >= CONFIG['VIABLE_JOB_THRESHOLD']:
            return t
    return 65  # Never reaches threshold

block_access['time_to_viable'] = block_access.apply(compute_time_tax, axis=1)
block_access['time_tax_minutes'] = block_access['time_to_viable'] - 5
block_access['never_viable'] = (block_access['time_to_viable'] == 65).astype(int)

# --- Aggregate to tract level ---
agg_dict = {
    'auto_jobs_30': ['mean', 'std'],
    'transit_jobs_30': ['mean', 'std'],
    'bike_safe_jobs_30': ['mean', 'std'],
    'walk_jobs_20': ['mean', 'std'],
    'time_tax_minutes': ['mean', 'median'],
    'time_to_viable': ['mean', 'median'],
    'never_viable': ['mean', 'sum'],     # proportion + count
    'block_id': 'count',                  # blocks per tract
}

tract_access = block_access.groupby('tract_geoid').agg(agg_dict)
tract_access.columns = ['_'.join(col).strip('_') for col in tract_access.columns]
tract_access = tract_access.rename(columns={
    'block_id_count': 'block_count',
    'never_viable_mean': 'pct_blocks_never_viable',
    'never_viable_sum': 'n_blocks_never_viable',
})
tract_access = tract_access.reset_index()

# Fill NaN std with 0 (tracts with 1 block)
for col in tract_access.columns:
    if col.endswith('_std'):
        tract_access[col] = tract_access[col].fillna(0)

print(f"\nTract-level accessibility: {len(tract_access)} tracts")
print(f"Columns: {list(tract_access.columns)}")
print(f"\nKey stats:")
print(f"  Auto jobs (mean per tract): median={tract_access['auto_jobs_30_mean'].median():.0f}")
print(f"  Transit jobs (mean per tract): median={tract_access['transit_jobs_30_mean'].median():.0f}")
print(f"  Bike safe jobs (mean per tract): median={tract_access['bike_safe_jobs_30_mean'].median():.0f}")
print(f"  Walk jobs (mean per tract): median={tract_access['walk_jobs_20_mean'].median():.0f}")
print(f"  Time tax (mean per tract): median={tract_access['time_tax_minutes_mean'].median():.0f} min")
print(f"  Blocks per tract: median={tract_access['block_count'].median():.0f}, range=[{tract_access['block_count'].min()}, {tract_access['block_count'].max()}]")


### 2.4 GTFS Transit Schedule Data
Loading stops (for spatial join), stop_times (for temporal analysis), trips and routes (for service metrics).


In [ ]:
# Load GTFS sheets
print("Loading GTFS data from transit_data.xlsx...")
gtfs_stops = pd.read_excel(PATHS['gtfs'], sheet_name='stops')
print(f"  Stops: {len(gtfs_stops)} rows")

gtfs_trips = pd.read_excel(PATHS['gtfs'], sheet_name='trips')
print(f"  Trips: {len(gtfs_trips)} rows")

gtfs_routes = pd.read_excel(PATHS['gtfs'], sheet_name='routes')
print(f"  Routes: {len(gtfs_routes)} rows")

gtfs_calendar = pd.read_excel(PATHS['gtfs'], sheet_name='calendar')
print(f"  Calendar: {len(gtfs_calendar)} rows")

print("\nLoading stop_times (this may take 30-60 seconds)...")
gtfs_stop_times = pd.read_excel(PATHS['gtfs'], sheet_name='stop_times')
print(f"  Stop_times: {len(gtfs_stop_times)} rows")

# Parse departure_time to minutes since midnight
def parse_gtfs_time_to_minutes(time_str):
    """Convert HH:MM:SS (possibly >24:00:00) to minutes since midnight."""
    try:
        parts = str(time_str).split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return np.nan

gtfs_stop_times['dep_minutes'] = gtfs_stop_times['departure_time'].apply(parse_gtfs_time_to_minutes)
gtfs_stop_times['dep_hour'] = (gtfs_stop_times['dep_minutes'] // 60).astype('Int64')
print(f"  Departure hours range: {gtfs_stop_times['dep_hour'].min()} to {gtfs_stop_times['dep_hour'].max()}")

print("\nGTFS loading complete.")


### 2.5 GTFS Spatial Join → Block → Tract Aggregation

1. Assign each stop to a census block (point-in-polygon)
2. Map blocks to tracts (first 11 digits of block GEOID)
3. Aggregate GTFS service metrics to tract level:
   - **Sum** for count measures: stops, unique routes, trips
   - **Mean** for proportions: wheelchair accessibility
   - **Sum by time window** for temporal analysis: peak/midday/evening/late trips


In [ ]:
# Create GeoDataFrame of stops
stops_gdf = gpd.GeoDataFrame(
    gtfs_stops,
    geometry=gpd.points_from_xy(gtfs_stops['stop_lon'], gtfs_stops['stop_lat']),
    crs='EPSG:4326'
)

# Ensure blocks are in same CRS
if blocks_gdf.crs != stops_gdf.crs:
    blocks_gdf = blocks_gdf.to_crs(stops_gdf.crs)

# Spatial join: which block does each stop fall in?
stops_in_blocks = gpd.sjoin(stops_gdf, blocks_gdf[['block_id', 'tract_geoid', 'geometry']],
                             how='left', predicate='within')

matched = stops_in_blocks['block_id'].notna().sum()
print(f"Spatial join: {matched}/{len(stops_gdf)} stops matched ({matched/len(stops_gdf)*100:.1f}%)")

# Save stop-to-block-to-tract mapping
stop_block_map = stops_in_blocks[['stop_id', 'block_id', 'tract_geoid']].dropna(subset=['block_id']).copy()
stop_block_map['block_id'] = stop_block_map['block_id'].astype(str)
stop_block_map['tract_geoid'] = stop_block_map['tract_geoid'].astype(str)

# --- Build GTFS metrics at BLOCK level first, then aggregate to tract ---
# Merge stop_times with trip info to get route_id
st_with_route = gtfs_stop_times.merge(gtfs_trips[['trip_id', 'route_id']], on='trip_id', how='left')

# Merge with stop-block-tract mapping
st_with_block = st_with_route.merge(stop_block_map, on='stop_id', how='inner')

# --- Per-TRACT aggregations ---
# Stops per tract
tract_stops = stop_block_map.groupby('tract_geoid')['stop_id'].nunique().rename('stop_count')

# Routes per tract
tract_routes = st_with_block.groupby('tract_geoid')['route_id'].nunique().rename('route_count')

# Trips per tract
tract_trips = st_with_block.groupby('tract_geoid')['trip_id'].nunique().rename('daily_trip_count')

# Wheelchair accessibility per tract (mean across stops)
stop_wheelchair = gtfs_stops[['stop_id', 'wheelchair_boarding']].copy()
stop_wheelchair['is_accessible'] = stop_wheelchair['wheelchair_boarding'].isin([1, '1'])
stops_with_access = stop_block_map.merge(stop_wheelchair[['stop_id', 'is_accessible']], on='stop_id')
tract_wheelchair = stops_with_access.groupby('tract_geoid')['is_accessible'].mean().rename('wheelchair_pct')

# --- Temporal trip counts per TRACT ---
peak_start, peak_end = CONFIG['PEAK_HOURS']
mid_start, mid_end = CONFIG['MIDDAY_HOURS']
eve_start, eve_end = CONFIG['EVENING_HOURS']
late_start, late_end = CONFIG['LATE_HOURS']

st_with_block['is_peak'] = st_with_block['dep_hour'].between(peak_start, peak_end - 1)
st_with_block['is_midday'] = st_with_block['dep_hour'].between(mid_start, mid_end - 1)
st_with_block['is_evening'] = st_with_block['dep_hour'].between(eve_start, eve_end - 1)
st_with_block['is_late'] = st_with_block['dep_hour'].between(late_start, late_end - 1)

tract_peak_trips = st_with_block[st_with_block['is_peak']].groupby('tract_geoid')['trip_id'].nunique().rename('peak_trips')
tract_midday_trips = st_with_block[st_with_block['is_midday']].groupby('tract_geoid')['trip_id'].nunique().rename('midday_trips')
tract_evening_trips = st_with_block[st_with_block['is_evening']].groupby('tract_geoid')['trip_id'].nunique().rename('evening_trips')
tract_late_trips = st_with_block[st_with_block['is_late']].groupby('tract_geoid')['trip_id'].nunique().rename('late_trips')

# Combine all GTFS metrics at tract level
gtfs_tract = pd.DataFrame(index=tract_stops.index)
for s in [tract_stops, tract_routes, tract_trips, tract_wheelchair,
          tract_peak_trips, tract_midday_trips, tract_evening_trips, tract_late_trips]:
    gtfs_tract = gtfs_tract.join(s, how='outer')

gtfs_tract = gtfs_tract.fillna(0)
gtfs_tract.index.name = 'tract_geoid'
gtfs_tract = gtfs_tract.reset_index()

print(f"\nGTFS metrics at tract level: {len(gtfs_tract)} tracts with transit service")
print(f"Metric distributions:")
for col in ['stop_count', 'route_count', 'daily_trip_count', 'peak_trips']:
    print(f"  {col}: median={gtfs_tract[col].median():.0f}, max={gtfs_tract[col].max():.0f}")


### 2.6 Census Tract Crosswalk: 2010 → 2020

**Problem:** GeoPackage block GEOIDs encode **2010 Census tract numbers** (the employment data is 2019 LODES, built on 2010 block geography). ACS 2023 5-Year uses **2020 Census tract boundaries**. Between 2010 and 2020, tracts were split, merged, and renumbered. Without a crosswalk, only ~67% of our tracts match ACS data.

**Solution:** The Census Bureau publishes a tract relationship file (`tab20_tract20_tract10_st12.txt`) that maps every 2020 tract to its corresponding 2010 tract(s), with land area for handling splits.

**Approach:**
1. Load the crosswalk, filter to Miami-Dade (county 086)
2. For 2010 tracts that map 1:1 to a 2020 tract → simple rename
3. For 2010 tracts that split into multiple 2020 tracts → assign the 2020 tract with the largest land area overlap (the primary successor)
4. Apply the mapping to translate our 2010-based tract_geoid to 2020 equivalents before merging with ACS


In [ ]:
# Load the Census Bureau tract relationship file
# Columns: GEOID_TRACT_20, GEOID_TRACT_10, AREALAND_PART (land area of overlap)
xwalk = pd.read_csv(PATHS['tract_crosswalk'], sep='|', dtype=str)

# Filter to Miami-Dade county (FIPS 12086) on the 2010 side
xwalk_md = xwalk[xwalk['GEOID_TRACT_10'].str[:5] == '12086'].copy()
xwalk_md['AREALAND_PART'] = pd.to_numeric(xwalk_md['AREALAND_PART'], errors='coerce').fillna(0)

print(f"Crosswalk loaded: {len(xwalk)} rows total, {len(xwalk_md)} Miami-Dade rows")
print(f"  Unique 2010 tracts: {xwalk_md['GEOID_TRACT_10'].nunique()}")
print(f"  Unique 2020 tracts: {xwalk_md['GEOID_TRACT_20'].nunique()}")

# For each 2010 tract, find its primary 2020 successor (largest land area overlap)
primary_2020 = (
    xwalk_md
    .sort_values('AREALAND_PART', ascending=False)
    .drop_duplicates(subset='GEOID_TRACT_10', keep='first')
    [['GEOID_TRACT_10', 'GEOID_TRACT_20', 'AREALAND_PART']]
    .copy()
)
primary_2020.columns = ['tract_2010', 'tract_2020', 'overlap_area']

# Check: how many are simple 1:1 vs splits?
tracts_per_2010 = xwalk_md.groupby('GEOID_TRACT_10')['GEOID_TRACT_20'].nunique()
n_one_to_one = (tracts_per_2010 == 1).sum()
n_splits = (tracts_per_2010 > 1).sum()
print(f"\n  1:1 mappings (unchanged or simple rename): {n_one_to_one}")
print(f"  Splits (1 2010 tract → multiple 2020 tracts): {n_splits}")

# Apply to our tract_access DataFrame
tract_access['tract_2010'] = tract_access['tract_geoid']  # preserve original
tract_access = tract_access.merge(primary_2020[['tract_2010', 'tract_2020']], on='tract_2010', how='left')

# For tracts already in 2020 format (matched directly) or outside Miami-Dade, keep as-is
tract_access['tract_geoid_2020'] = tract_access['tract_2020'].fillna(tract_access['tract_geoid'])

mapped = tract_access['tract_2020'].notna().sum()
print(f"\nCrosswalk applied to tract_access:")
print(f"  Mapped via crosswalk: {mapped}/{len(tract_access)} ({mapped/len(tract_access)*100:.1f}%)")
print(f"  Kept original (non-Miami-Dade edge tracts): {len(tract_access) - mapped}")

# Also apply crosswalk to GTFS tract metrics
gtfs_tract['tract_2010'] = gtfs_tract['tract_geoid']
gtfs_tract = gtfs_tract.merge(primary_2020[['tract_2010', 'tract_2020']], on='tract_2010', how='left')
gtfs_tract['tract_geoid_2020'] = gtfs_tract['tract_2020'].fillna(gtfs_tract['tract_geoid'])

# Verify against ACS
acs_geoids = set(acs['GEOID'].astype(str).str.zfill(11))
translated_geoids = set(tract_access['tract_geoid_2020'])
match_after = len(translated_geoids & acs_geoids)
print(f"\nPre-crosswalk ACS match: {len(set(tract_access['tract_2010']) & acs_geoids)}/{len(tract_access)} tracts")
print(f"Post-crosswalk ACS match: {match_after}/{len(tract_access)} tracts")


### 2.7 Master Tract-Level DataFrame

Merge all three data sources into a single tract-level DataFrame:
- ACS demographics (707 tracts, 2020 Census geography)
- GeoPackage accessibility (aggregated from blocks, ~522 tracts, translated to 2020 IDs via crosswalk)
- GTFS service metrics (aggregated from stops → blocks → tracts, translated to 2020 IDs)

**Merge strategy:** Start with GeoPackage tracts (our study area), merge ACS on the 2020-translated tract GEOID.
Unmatched tracts are **dropped** (not imputed): county medians carry no signal for transit inequity analysis.
Dropped tracts are either (a) non-Miami-Dade GeoPackage edge artifacts or (b) 98xx institutional/special-tabulation tracts — neither represents a valid residential unit for policy recommendations.


In [ ]:
# Start with tract-level accessibility, using 2020-translated GEOIDs
master = tract_access.copy()
master['tract_geoid'] = master['tract_geoid_2020']  # Use 2020 IDs for ACS merge

# Merge ACS on 2020 tract GEOIDs
acs_for_merge = acs.copy()
acs_for_merge = acs_for_merge.rename(columns={'GEOID': 'tract_geoid'})
acs_for_merge['tract_geoid'] = acs_for_merge['tract_geoid'].astype(str).str.zfill(11)

master = master.merge(acs_for_merge, on='tract_geoid', how='left')
acs_matched = master['poverty_rate_pct'].notna().sum()
print(f"ACS merge (with crosswalk): {acs_matched}/{len(master)} tracts matched ({acs_matched/len(master)*100:.1f}%)")

# Merge GTFS tract metrics (also using 2020 IDs)
gtfs_for_merge = gtfs_tract.copy()
gtfs_for_merge['tract_geoid'] = gtfs_for_merge['tract_geoid_2020']
# Drop duplicate columns before merge
gtfs_for_merge = gtfs_for_merge.drop(columns=['tract_2010', 'tract_2020', 'tract_geoid_2020'], errors='ignore')
master = master.merge(gtfs_for_merge, on='tract_geoid', how='left')
gtfs_cols = ['stop_count', 'route_count', 'daily_trip_count', 'wheelchair_pct',
             'peak_trips', 'midday_trips', 'evening_trips', 'late_trips']
for col in gtfs_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

# --- QUALITY FILTER: Drop unmatched tracts (no real ACS data) ---
# These fall into two categories:
#   1. Non-Miami-Dade tracts (Broward/Monroe, county prefix != 12086): GeoPackage edge artifacts, outside study area
#   2. Unmatched Miami-Dade tracts: all happen to be 98xx institutional/special-tabulation tracts
#      (group quarters, correctional facilities) — already flagged, not valid for residential transit analysis
# County medians are analytically useless for our transit inequity composite, so we drop rather than impute.
unmatched_mask = master['poverty_rate_pct'].isna()
unmatched_tracts = master.loc[unmatched_mask, ['tract_geoid', 'tract_2010']].copy()
unmatched_tracts['county_fips'] = unmatched_tracts['tract_geoid'].str[:5]
unmatched_tracts['reason'] = np.where(
    unmatched_tracts['county_fips'] != '12086',
    'Non-Miami-Dade (out-of-study-area)',
    'No ACS match (institutional/special-tabulation tract)'
)
print(f"\n--- QUALITY FILTER: Dropping {unmatched_mask.sum()} unmatched tracts ---")
print(unmatched_tracts[['tract_geoid', 'county_fips', 'reason']].to_string(index=False))

master = master[~unmatched_mask].copy()
print(f"\nStudy-area tracts after filter: {len(master)} (all Miami-Dade, all with real ACS data)")
print(f"  Dropped: {unmatched_mask.sum()} ({unmatched_mask.sum()/len(unmatched_mask)*100:.1f}% of raw)")

# Derive transit_desert flag at tract level
master['transit_desert'] = (master['transit_jobs_30_mean'] == 0).astype(int)

print(f"\nMaster DataFrame: {master.shape[0]} tracts × {master.shape[1]} columns")
print(f"  Transit deserts: {master['transit_desert'].sum()} tracts ({master['transit_desert'].mean()*100:.1f}%)")
print(f"  Tracts with GTFS stops: {(master['stop_count'] > 0).sum()}")


## 3. Compute Composite Equity Indicators

### Indicator 1: Transit Dependency Index
**Question:** Which tracts have residents who NEED transit but CANNOT reach jobs through any non-auto mode?

**Logic:** Need (no-vehicle + poverty) × Access Deficit (low transit + low safe bike + low walk-to-job access)

**Walk note:** The 20-minute walk threshold measures walking as a *direct commute to the job*. Walking 20 min to a transit stop defeats the purpose — that's addressed in Service Coverage.


In [ ]:
# --- Need Score ---
master['need_novehicle_norm'] = normalize_01(master['hh_no_vehicle_pct'].fillna(0))
master['need_poverty_norm'] = normalize_01(master['poverty_rate_pct'].fillna(0))
master['need_score'] = (master['need_novehicle_norm'] + master['need_poverty_norm']) / 2

# --- Access Deficit ---
# Using tract-level mean accessibility (aggregated from blocks)
master['deficit_transit'] = invert_normalize(master['transit_jobs_30_mean'].fillna(0))
master['deficit_bike_safe'] = invert_normalize(master['bike_safe_jobs_30_mean'].fillna(0))
master['deficit_walk'] = invert_normalize(master['walk_jobs_20_mean'].fillna(0))

w_t = CONFIG['TRANSIT_WEIGHT']  # 0.5
w_b = CONFIG['BIKE_WEIGHT']     # 0.3
w_w = CONFIG['WALK_WEIGHT']     # 0.2

master['access_deficit_1'] = (
    w_t * master['deficit_transit'] +
    w_b * master['deficit_bike_safe'] +
    w_w * master['deficit_walk']
)

# --- Transit Dependency Index = Need × Access Deficit ---
master['ind_1_transit_dependency'] = master['need_score'] * master['access_deficit_1']

print("Indicator 1: Transit Dependency Index")
print(f"  Need score:     median={master['need_score'].median():.3f}, mean={master['need_score'].mean():.3f}")
print(f"  Access deficit:  median={master['access_deficit_1'].median():.3f}, mean={master['access_deficit_1'].mean():.3f}")
print(f"  Transit Dep:     median={master['ind_1_transit_dependency'].median():.3f}, std={master['ind_1_transit_dependency'].std():.3f}")
print(f"  Range: [{master['ind_1_transit_dependency'].min():.4f}, {master['ind_1_transit_dependency'].max():.4f}]")


### Indicator 2: Temporal Service Mismatch
**Question:** Which tracts look served during peak hours but fail people who depend on transit at off-peak times?

**Logic:** Peak-to-offpeak service ratio × shift-worker proxy (low income ≈ shift workers)

**Three-tier approach (from v2 fix):**
1. Tracts WITH GTFS service: peak/offpeak ratio × shift-worker proxy
2. Tracts with transit accessibility but no stops: moderate baseline mismatch × proxy
3. Tracts with no transit at all: low baseline × proxy


In [ ]:
# --- Shift-worker proxy ---
master['low_income_share'] = (
    master['hh_income_under_10k_pct'].fillna(0) +
    master['hh_income_10k_15k_pct'].fillna(0) +
    master['hh_income_15k_25k_pct'].fillna(0)
)

master['shift_worker_proxy'] = np.where(
    (master['poverty_rate_pct'].fillna(0) > CONFIG['POVERTY_THRESHOLD']) |
    (master['low_income_share'] > CONFIG['LOW_INCOME_BUCKET_THRESHOLD']),
    1.0, 0.3
)

# --- Temporal ratio ---
master['offpeak_avg_trips'] = (master['midday_trips'] + master['evening_trips'] + master['late_trips']) / 3

master['peak_offpeak_ratio'] = np.where(
    master['offpeak_avg_trips'] > 0,
    master['peak_trips'] / master['offpeak_avg_trips'],
    np.where(master['peak_trips'] > 0, 5.0, 0)
)
# Cap at 99th percentile of served tracts
ratio_cap = np.percentile(
    master.loc[master['peak_trips'] > 0, 'peak_offpeak_ratio'].dropna(), 99
) if (master['peak_trips'] > 0).any() else 5.0
master['peak_offpeak_ratio'] = master['peak_offpeak_ratio'].clip(upper=ratio_cap)

# --- Three-tier mismatch ---
has_gtfs = master['stop_count'] > 0
has_transit_access = master['transit_jobs_30_mean'] > 0

gtfs_mismatch = normalize_01(master['peak_offpeak_ratio']) * master['shift_worker_proxy']
proximity_mismatch = 0.5 * master['shift_worker_proxy']
no_service_mismatch = 0.2 * master['shift_worker_proxy']

master['ind_2_temporal_mismatch'] = np.where(
    has_gtfs, gtfs_mismatch,
    np.where(has_transit_access, proximity_mismatch, no_service_mismatch)
)
master['ind_2_temporal_mismatch'] = normalize_01(master['ind_2_temporal_mismatch'])

print("Indicator 2: Temporal Service Mismatch")
print(f"  Tracts with GTFS stops: {has_gtfs.sum()} ({has_gtfs.mean()*100:.1f}%)")
print(f"  Tracts near transit (no stops): {(~has_gtfs & has_transit_access).sum()}")
print(f"  No transit at all: {(~has_transit_access).sum()}")
print(f"  Shift-worker proxy=1.0: {(master['shift_worker_proxy'] == 1.0).sum()} tracts ({(master['shift_worker_proxy'] == 1.0).mean()*100:.1f}%)")
print(f"  Temporal Mismatch: median={master['ind_2_temporal_mismatch'].median():.3f}, std={master['ind_2_temporal_mismatch'].std():.3f}")


### Indicator 3: Structural Access Gap
**Question:** How much worse is life without a car, measured in reachable jobs?

**Logic:** log2(auto_jobs_30 / transit_jobs_30) — purely structural, no demographics.

Sprint 1 found a 107.6x median gap. Using log2 avoids extreme outlier distortion.
Now computed on tract-level mean accessibility.


In [ ]:
# Avoid log(0): replace zero transit with 1
transit_safe = master['transit_jobs_30_mean'].replace(0, 1)
auto_safe = master['auto_jobs_30_mean'].replace(0, 1)

master['structural_gap_log2'] = np.log2(auto_safe / transit_safe)

master['ind_3_structural_gap'] = normalize_01(master['structural_gap_log2'])

print("Indicator 3: Structural Access Gap")
print(f"  Raw log2 gap: median={master['structural_gap_log2'].median():.1f}, mean={master['structural_gap_log2'].mean():.1f}")
print(f"  Interpretation: median gap = {2**master['structural_gap_log2'].median():.0f}x more jobs by car")
print(f"  Transit deserts: {master['transit_desert'].sum()} tracts ({master['transit_desert'].mean()*100:.1f}%)")
print(f"  Normalized gap: median={master['ind_3_structural_gap'].median():.3f}, std={master['ind_3_structural_gap'].std():.3f}")


### Indicator 4: Time Tax
**Question:** How long does it take transit to provide *meaningful* job access in each tract?

**Revised logic (from v2):** Time Tax = mean minutes by transit to reach a viable employment threshold (10,000 jobs). Computed at block level then averaged to tract, preserving granular transit time curves.

Blocks that never reach the threshold get 65-minute penalty. The tract-level score is the mean time tax across all blocks in the tract.


In [ ]:
# Time tax was computed at block level and aggregated in Cell 6
# tract_access already has: time_tax_minutes_mean, time_tax_minutes_median, pct_blocks_never_viable

master['ind_4_time_tax'] = normalize_01(master['time_tax_minutes_mean'])

print("Indicator 4: Time Tax (revised — viable job threshold)")
print(f"  Viable threshold: {CONFIG['VIABLE_JOB_THRESHOLD']:,} jobs by transit")
print(f"  Tract mean time tax: median={master['time_tax_minutes_mean'].median():.0f} min, mean={master['time_tax_minutes_mean'].mean():.0f} min")
print(f"  % blocks never viable (tract avg): median={master['pct_blocks_never_viable'].median()*100:.0f}%")
print(f"  Normalized: median={master['ind_4_time_tax'].median():.3f}, std={master['ind_4_time_tax'].std():.3f}")


### Indicator 5: Service Coverage
**Question:** How well does the physical transit network actually reach each tract?

**Two-tier approach (from v2 fix):**
1. **Tracts WITH stops:** Weighted deficit of stop density, route diversity, trip frequency, wheelchair access
2. **Tracts WITHOUT stops:** Use mean transit_jobs as proxy — if transit accessible, stops are nearby. If zero transit access, true desert.

At tract level, this is cleaner: we know exactly which tracts have stops and which don't.


In [ ]:
# --- For tracts WITH stops: weighted service metrics ---
master['svc_deficit_stops'] = invert_normalize(master['stop_count'])
master['svc_deficit_routes'] = invert_normalize(master['route_count'])
master['svc_deficit_trips'] = invert_normalize(master['daily_trip_count'])
master['svc_deficit_wheelchair'] = invert_normalize(master['wheelchair_pct'])

w = CONFIG['SVC_WEIGHTS']
gtfs_deficit = (
    w['stops'] * master['svc_deficit_stops'] +
    w['routes'] * master['svc_deficit_routes'] +
    w['trips'] * master['svc_deficit_trips'] +
    w['wheelchair'] * master['svc_deficit_wheelchair']
)

# --- For tracts WITHOUT stops: transit proximity proxy ---
transit_proxy_deficit = invert_normalize(master['transit_jobs_30_mean'].fillna(0))

has_stops = master['stop_count'] > 0
master['ind_5_service_coverage'] = np.where(
    has_stops,
    gtfs_deficit * 0.7,
    np.where(
        master['transit_jobs_30_mean'] > 0,
        0.5 + 0.5 * transit_proxy_deficit,
        1.0  # True transit desert
    )
)

true_deserts = (master['stop_count'] == 0) & (master['transit_jobs_30_mean'] == 0)

print("Indicator 5: Service Coverage (Deficit)")
print(f"  Tracts with stops: {has_stops.sum()} ({has_stops.mean()*100:.1f}%)")
print(f"  Tracts with no stops but transit accessible: {(~has_stops & (master['transit_jobs_30_mean'] > 0)).sum()}")
print(f"  True transit deserts: {true_deserts.sum()} ({true_deserts.mean()*100:.1f}%)")
print(f"  Service deficit: median={master['ind_5_service_coverage'].median():.3f}, std={master['ind_5_service_coverage'].std():.3f}")


### Indicator 6: Economic Vulnerability
**Question:** How economically fragile are the residents, independent of transit access?

**Logic:** Weighted average of poverty (0.30), SNAP (0.25), rent burden (0.25), unemployment (0.20).

This is the demand side — how much do residents NEED affordable transit? At tract level, this is native ACS data with no aggregation artifacts.


In [ ]:
w_e = CONFIG['ECON_WEIGHTS']

master['econ_poverty_norm'] = normalize_01(master['poverty_rate_pct'].fillna(0))
master['econ_snap_norm'] = normalize_01(master['snap_benefits_pct'].fillna(0))
master['econ_rent_norm'] = normalize_01(master['rent_burden_50pct_plus'].fillna(0))
master['econ_unemp_norm'] = normalize_01(master['unemployment_rate_pct'].fillna(0))

master['ind_6_economic_vulnerability'] = (
    w_e['poverty'] * master['econ_poverty_norm'] +
    w_e['snap'] * master['econ_snap_norm'] +
    w_e['rent_burden'] * master['econ_rent_norm'] +
    w_e['unemployment'] * master['econ_unemp_norm']
)

print("Indicator 6: Economic Vulnerability")
print(f"  Vulnerability: median={master['ind_6_economic_vulnerability'].median():.3f}, std={master['ind_6_economic_vulnerability'].std():.3f}")
print(f"  Range: [{master['ind_6_economic_vulnerability'].min():.4f}, {master['ind_6_economic_vulnerability'].max():.4f}]")
print(f"  Top 10% threshold: {master['ind_6_economic_vulnerability'].quantile(0.90):.3f}")


### Indicator 7: Multimodal Accessibility (Deficit)
**Question:** Can people realistically combine transit, safe biking, and walking to reach jobs?

**Logic:** Weighted score of transit_30 (0.5) + bike_safe_30 (0.3) + walk_to_job_20 (0.2). Deficit = 1 - score.

Uses LTS1 (safe bike routes) only. Sprint 1 showed 32.8x gap between LTS4 and LTS1.


In [ ]:
# Using tract-level mean accessibility
master['multi_transit_norm'] = normalize_01(master['transit_jobs_30_mean'].fillna(0))
master['multi_bike_norm'] = normalize_01(master['bike_safe_jobs_30_mean'].fillna(0))
master['multi_walk_norm'] = normalize_01(master['walk_jobs_20_mean'].fillna(0))

multimodal_score = (
    CONFIG['TRANSIT_WEIGHT'] * master['multi_transit_norm'] +
    CONFIG['BIKE_WEIGHT'] * master['multi_bike_norm'] +
    CONFIG['WALK_WEIGHT'] * master['multi_walk_norm']
)

master['ind_7_multimodal_deficit'] = 1.0 - multimodal_score

desert_threshold = master['ind_7_multimodal_deficit'].quantile(0.90)
master['multimodal_desert'] = (master['ind_7_multimodal_deficit'] >= desert_threshold).astype(int)

print("Indicator 7: Multimodal Accessibility (Deficit)")
print(f"  Multimodal score: median={multimodal_score.median():.3f}")
print(f"  Multimodal deficit: median={master['ind_7_multimodal_deficit'].median():.3f}, std={master['ind_7_multimodal_deficit'].std():.3f}")
print(f"  Multimodal deserts (top 10% deficit): {master['multimodal_desert'].sum()} tracts")


## 4. Final Composite: Equity Priority Score

**Formula:** `Equity Priority = Need × Access Deficit` (multiplicative, not averaged)

**Need** = average of Transit Dependency (Ind 1) and Economic Vulnerability (Ind 6)
**Access Deficit** = average of Structural Gap (3), Time Tax (4), Service Coverage (5), Temporal Mismatch (2), Multimodal Deficit (7)

**Why multiplicative?**
- Wealthy area + bad transit = Low priority (they have alternatives)
- Poor area + good transit = Low priority (system works for them)
- Poor area + bad transit = **HIGH priority** (this is where intervention matters)


In [ ]:
# --- Need Component ---
master['composite_need'] = (
    master['ind_1_transit_dependency'] + master['ind_6_economic_vulnerability']
) / 2

# --- Access Deficit Component ---
master['composite_access_deficit'] = (
    master['ind_3_structural_gap'] +
    master['ind_4_time_tax'] +
    master['ind_5_service_coverage'] +
    master['ind_2_temporal_mismatch'] +
    master['ind_7_multimodal_deficit']
) / 5

# --- Equity Priority Score = Need × Access Deficit ---
master['equity_priority_score'] = master['composite_need'] * master['composite_access_deficit']

# --- Tier Assignment ---
tiers = CONFIG['TIER_PERCENTILES']
p90 = master['equity_priority_score'].quantile(tiers[0] / 100)
p70 = master['equity_priority_score'].quantile(tiers[1] / 100)
p40 = master['equity_priority_score'].quantile(tiers[2] / 100)

master['equity_tier'] = pd.cut(
    master['equity_priority_score'],
    bins=[-np.inf, p40, p70, p90, np.inf],
    labels=['Low', 'Moderate', 'High', 'Critical']
)

master['equity_percentile'] = master['equity_priority_score'].rank(pct=True) * 100

print("=== EQUITY PRIORITY SCORE ===")
print(f"  Need:     median={master['composite_need'].median():.3f}, mean={master['composite_need'].mean():.3f}")
print(f"  Deficit:  median={master['composite_access_deficit'].median():.3f}, mean={master['composite_access_deficit'].mean():.3f}")
print(f"  Priority: median={master['equity_priority_score'].median():.4f}, mean={master['equity_priority_score'].mean():.4f}")
print(f"  Range:    [{master['equity_priority_score'].min():.6f}, {master['equity_priority_score'].max():.6f}]")
print(f"\nTier distribution:")
print(master['equity_tier'].value_counts().sort_index())
print(f"\nPercentile cutoffs: Critical>{p90:.4f}, High>{p70:.4f}, Moderate>{p40:.4f}")


## 5. Validation and Sanity Checks

### 5.1 Indicator Distributions
Each indicator should produce a meaningful 0-1 distribution with variance. At tract level we expect smoother distributions (averaging smooths block-level extremes).


In [ ]:
indicator_cols = [
    ('ind_1_transit_dependency', 'Transit Dependency'),
    ('ind_2_temporal_mismatch', 'Temporal Mismatch'),
    ('ind_3_structural_gap', 'Structural Access Gap'),
    ('ind_4_time_tax', 'Time Tax'),
    ('ind_5_service_coverage', 'Service Coverage Deficit'),
    ('ind_6_economic_vulnerability', 'Economic Vulnerability'),
    ('ind_7_multimodal_deficit', 'Multimodal Deficit'),
    ('equity_priority_score', 'EQUITY PRIORITY SCORE'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (col, title) in enumerate(indicator_cols):
    ax = axes[i]
    master[col].hist(bins=50, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Score (0-1)')
    ax.axvline(master[col].median(), color='red', linestyle='--', label=f'median={master[col].median():.3f}')
    ax.legend(fontsize=7)

plt.suptitle('Indicator Distributions — Tract Level (0-1 Scale)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('indicator_distributions_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Distributions saved to indicator_distributions_v3.png")


### 5.2 Correlation Matrix
We expect moderate correlations between indicators (they measure related but distinct aspects).


In [ ]:
ind_cols = [col for col, _ in indicator_cols[:7]]
corr = master[ind_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0,
            xticklabels=[t for _, t in indicator_cols[:7]],
            yticklabels=[t for _, t in indicator_cols[:7]],
            ax=ax, vmin=-0.3, vmax=1)
ax.set_title('Indicator Correlation Matrix — Tract Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('indicator_correlations_v3.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelation insights:")
for i in range(len(ind_cols)):
    for j in range(i+1, len(ind_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            print(f"  HIGH ({r:.2f}): {indicator_cols[i][1]} <-> {indicator_cols[j][1]}")
        elif abs(r) < 0.1:
            print(f"  LOW ({r:.2f}): {indicator_cols[i][1]} <-> {indicator_cols[j][1]}")


### 5.3 Neighborhood Sanity Check

**Approach:** Dissolve block geometries into tract polygons, then use a systematic grid of lat/lon points
within each neighborhood's geographic extent to find which tracts fall within that area.
This uses our actual GeoPackage spatial data — no hardcoded tract IDs.

**Note on Overtown:** Overtown is located near downtown Miami with direct Metrorail access and high
transit job accessibility (170K-235K jobs reachable by transit in 30 min). The composite correctly
scores it lower because Need × Access Deficit yields low priority when access is already good.
This is a feature, not a bug — it means transit is actually serving Overtown relatively well.


In [ ]:
# --- Build tract polygons from block geometries ---
print("Dissolving blocks to tract polygons for spatial neighborhood lookup...")
tracts_geo = blocks_gdf[['tract_geoid', 'geometry']].dissolve(by='tract_geoid').reset_index()
if tracts_geo.crs and tracts_geo.crs.to_epsg() != 4326:
    tracts_geo = tracts_geo.to_crs(epsg=4326)

# --- Systematic grid search to find tracts in each neighborhood ---
# Neighborhood geographic extents (lat/lon bounding boxes)
# These represent known geographic areas — the spatial query finds which of our
# 522 tracts actually fall within each area.
neighborhood_extents = {
    'Overtown':      {'lat': (25.773, 25.792), 'lon': (-80.210, -80.190)},
    'Liberty City':  {'lat': (25.808, 25.840), 'lon': (-80.245, -80.205)},
    'Little Haiti':  {'lat': (25.825, 25.855), 'lon': (-80.200, -80.175)},
    'Brownsville':   {'lat': (25.810, 25.832), 'lon': (-80.255, -80.230)},
    'Homestead':     {'lat': (25.445, 25.490), 'lon': (-80.500, -80.440)},
    'Opa-locka':     {'lat': (25.885, 25.915), 'lon': (-80.265, -80.225)},
    'Allapattah':    {'lat': (25.793, 25.818), 'lon': (-80.235, -80.205)},
    'Coral Gables':  {'lat': (25.710, 25.760), 'lon': (-80.290, -80.245)},
    'Key Biscayne':  {'lat': (25.680, 25.710), 'lon': (-80.175, -80.148)},
    'Brickell':      {'lat': (25.752, 25.770), 'lon': (-80.200, -80.185)},
    'Downtown Miami':{'lat': (25.770, 25.790), 'lon': (-80.200, -80.185)},
}

from shapely.geometry import Point

def find_tracts_in_area(lat_range, lon_range, grid_size=0.002):
    lats = np.arange(lat_range[0], lat_range[1], grid_size)
    lons = np.arange(lon_range[0], lon_range[1], grid_size)
    found = set()
    for lat in lats:
        for lon in lons:
            matches = tracts_geo[tracts_geo.geometry.contains(Point(lon, lat))]
            if len(matches) > 0:
                found.update(matches['tract_geoid'].tolist())
    return sorted(found)

neighborhood_tracts = {}
for name, bounds in neighborhood_extents.items():
    neighborhood_tracts[name] = find_tracts_in_area(bounds['lat'], bounds['lon'])

# --- Flag institutional/special-purpose tracts ---
# Census 98xx tracts are special tabulation areas (group quarters, prisons, etc.)
master['institutional_tract'] = master['tract_geoid'].str[5:8].astype(int) >= 980
n_inst = master['institutional_tract'].sum()
print(f"Institutional tracts (98xx): {n_inst}")

# All remaining tracts have real ACS data (unmatched ones were dropped in quality filter above)
# No median imputation — every tract here has verified demographics
master['acs_matched'] = True  # All remaining tracts are ACS-matched by construction

print(f"\n=== NEIGHBORHOOD SANITY CHECK (from spatial lookup) ===")
print(f"{'Neighborhood':<20} {'N':>3} {'Mean':>8} {'Med':>8} {'Max':>8} {'Tiers':<35} {'Need':>7} {'Deficit':>8}")
print("-" * 105)

for name, tracts in neighborhood_tracts.items():
    mask = master['tract_geoid'].isin(tracts) & ~master['institutional_tract']
    n = mask.sum()
    if n > 0:
        sub = master.loc[mask]
        mean_s = sub['equity_priority_score'].mean()
        med_s = sub['equity_priority_score'].median()
        max_s = sub['equity_priority_score'].max()
        tiers = sub['equity_tier'].value_counts().to_dict()
        tier_str = ', '.join([f"{k}:{v}" for k, v in sorted(tiers.items())])
        need = sub['composite_need'].mean()
        deficit = sub['composite_access_deficit'].mean()
        print(f"  {name:<20} {n:>3} {mean_s:>8.4f} {med_s:>8.4f} {max_s:>8.4f} {tier_str:<35} {need:>7.3f} {deficit:>8.3f}")

print(f"\nTop 15 highest-priority tracts (excluding institutional):")
non_inst = master[~master['institutional_tract']]
top15 = non_inst.nlargest(15, 'equity_priority_score')[
    ['tract_geoid', 'equity_priority_score', 'equity_tier', 'composite_need', 'composite_access_deficit',
     'poverty_rate_pct', 'hh_no_vehicle_pct', 'transit_jobs_30_mean', 'stop_count']
]
print(top15.to_string(index=False))

print(f"\nNote: {n_inst} institutional (98xx) tracts excluded from sanity check.")
print(f"These are Census special-purpose tracts (group quarters, prisons, etc.)")


### 5.4 Summary Statistics

In [ ]:
all_ind_cols = [col for col, _ in indicator_cols]
summary = master[all_ind_cols].describe().T
summary['iqr'] = summary['75%'] - summary['25%']
summary.index = [t for _, t in indicator_cols]

print("=== INDICATOR SUMMARY STATISTICS (TRACT LEVEL) ===")
print(summary[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'iqr']].to_string())
print()

# Variance check
print("\n=== VARIANCE CHECK ===")
for col, name in indicator_cols[:7]:
    std = master[col].std()
    status = "OK" if std >= 0.05 else "LOW VARIANCE"
    pct_at_min = (master[col] == master[col].min()).mean() * 100
    pct_at_max = (master[col] == master[col].max()).mean() * 100
    print(f"  {name:<30} std={std:.3f} {status}  (min={pct_at_min:.1f}%, max={pct_at_max:.1f}%)")

print()
print("=== TIER DISTRIBUTION ===")
tier_dist = master['equity_tier'].value_counts().sort_index()
for tier, count in tier_dist.items():
    print(f"  {tier}: {count} tracts ({count/len(master)*100:.1f}%)")


## 6. Export

Save the tract-level dataset — this is now the primary output.
Also export a comparison table showing v2→v3 changes.


In [ ]:
# --- Tract-level export (primary output) ---
export_cols = ['tract_geoid', 'tract_2010', 'block_count',
    # Accessibility (tract means from block aggregation)
    'auto_jobs_30_mean', 'transit_jobs_30_mean', 'bike_safe_jobs_30_mean', 'walk_jobs_20_mean',
    'auto_jobs_30_std', 'transit_jobs_30_std',  # within-tract variation
    # Time Tax (aggregated from block-level computation)
    'time_tax_minutes_mean', 'time_tax_minutes_median', 'pct_blocks_never_viable',
    # GTFS
    'stop_count', 'route_count', 'daily_trip_count', 'peak_trips',
    'midday_trips', 'evening_trips', 'late_trips', 'wheelchair_pct',
    # ACS key vars (native tract-level)
    'poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
    'rent_burden_50pct_plus', 'unemployment_rate_pct',
    # 7 indicators
    'ind_1_transit_dependency', 'ind_2_temporal_mismatch',
    'ind_3_structural_gap', 'ind_4_time_tax',
    'ind_5_service_coverage', 'ind_6_economic_vulnerability',
    'ind_7_multimodal_deficit',
    # Composite
    'composite_need', 'composite_access_deficit',
    'equity_priority_score', 'equity_percentile', 'equity_tier',
    # Flags
    'transit_desert', 'multimodal_desert',
    'shift_worker_proxy', 'low_income_share', 'peak_offpeak_ratio',
    'institutional_tract',
]

# Only include columns that exist
export_cols = [c for c in export_cols if c in master.columns]
export_df = master[export_cols].copy()
export_df.to_csv('Sprint2_Equity_Indicators_v3_tract.csv', index=False)
print(f"Tract-level export: {export_df.shape[0]} rows × {export_df.shape[1]} columns")
print(f"  Saved to: Sprint2_Equity_Indicators_v3_tract.csv")

print(f"\n=== v2 → v3 CHANGE SUMMARY ===")
print(f"  Unit of analysis: Block (36,507) → Tract ({len(master)})")
print(f"  ACS merge: 100% real data (no imputation — unmatched tracts dropped)")
print(f"    Dropped: non-Miami-Dade edge tracts + institutional 98xx tracts")
print(f"  Accessibility: Block-level mean aggregated to tract")
print(f"  Time Tax: Block-level computation → tract-level mean")
print(f"  GTFS: Sum stops/routes/trips per tract")
print(f"  Indicator logic: Unchanged (same 7 formulas)")
print(f"  Composite: Unchanged (multiplicative Need × Deficit)")


## 6b. Results Workbook (Excel)

Export a 4-sheet Excel workbook for presentation and review:

1. **Equity Scores by Tract** — every tract with its 7 indicator scores + priority tier, sorted Critical→Low
2. **Summary by Tier** — tract counts, mean indicators, and mean ACS demographics per tier
3. **Top 20 Priority Tracts** — highest-priority tracts with full profile (indicators + ACS + GTFS)
4. **Methodology Reference** — what each indicator measures, its inputs, and composite weight


In [54]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# --- Color scheme ---
HEADER_FILL = PatternFill('solid', fgColor='1F4E79')
HEADER_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=10)
SUBHEADER_FILL = PatternFill('solid', fgColor='D6E4F0')
SUBHEADER_FONT = Font(name='Arial', bold=True, size=10)
DATA_FONT = Font(name='Arial', size=10)
TIER_COLORS = {
    'Critical': PatternFill('solid', fgColor='FF6B6B'),
    'High':     PatternFill('solid', fgColor='FFA94D'),
    'Moderate': PatternFill('solid', fgColor='FFD43B'),
    'Low':      PatternFill('solid', fgColor='69DB7C'),
}
THIN_BORDER = Border(
    bottom=Side(style='thin', color='B0B0B0')
)

def style_header(ws, row, num_cols):
    for col in range(1, num_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

def auto_width(ws, min_w=10, max_w=22):
    for col_cells in ws.columns:
        lengths = []
        for cell in col_cells:
            if cell.value:
                lengths.append(len(str(cell.value)))
        if lengths:
            w = min(max(max(lengths) + 2, min_w), max_w)
            ws.column_dimensions[get_column_letter(col_cells[0].column)].width = w

# ============================================================
# SHEET 1: Equity Scores by Tract
# ============================================================
ws1 = wb.active
ws1.title = 'Equity Scores by Tract'

tier_order = {'Critical': 0, 'High': 1, 'Moderate': 2, 'Low': 3}
sorted_df = master.copy()
sorted_df['_tier_sort'] = sorted_df['equity_tier'].map(tier_order)
sorted_df = sorted_df.sort_values(['_tier_sort', 'equity_priority_score'], ascending=[True, False])

s1_cols = ['tract_geoid', 'ind_1_transit_dependency', 'ind_2_temporal_mismatch',
           'ind_3_structural_gap', 'ind_4_time_tax', 'ind_5_service_coverage',
           'ind_6_economic_vulnerability', 'ind_7_multimodal_deficit',
           'equity_priority_score', 'equity_percentile', 'equity_tier']
s1_headers = ['Census Tract', 'Transit Dependency', 'Temporal Mismatch',
              'Structural Gap', 'Time Tax', 'Service Coverage',
              'Economic Vulnerability', 'Multimodal Deficit',
              'Priority Score', 'Percentile', 'Priority Tier']

ws1.append(s1_headers)
style_header(ws1, 1, len(s1_headers))

for _, row in sorted_df.iterrows():
    vals = [row[c] for c in s1_cols]
    ws1.append(vals)

for r in range(2, ws1.max_row + 1):
    tier_val = ws1.cell(row=r, column=len(s1_cols)).value
    for c in range(1, len(s1_cols) + 1):
        cell = ws1.cell(row=r, column=c)
        cell.font = DATA_FONT
        cell.border = THIN_BORDER
        if c >= 2 and c <= 9:
            cell.number_format = '0.000'
            cell.alignment = Alignment(horizontal='center')
        elif c == 10:
            cell.number_format = '0.0'
            cell.alignment = Alignment(horizontal='center')
        elif c == 11:
            cell.alignment = Alignment(horizontal='center')
    if tier_val in TIER_COLORS:
        ws1.cell(row=r, column=len(s1_cols)).fill = TIER_COLORS[tier_val]

ws1.freeze_panes = 'B2'
auto_width(ws1)

# ============================================================
# SHEET 2: Summary by Tier
# ============================================================
ws2 = wb.create_sheet('Summary by Tier')

indicator_cols = ['ind_1_transit_dependency', 'ind_2_temporal_mismatch', 'ind_3_structural_gap',
                  'ind_4_time_tax', 'ind_5_service_coverage', 'ind_6_economic_vulnerability',
                  'ind_7_multimodal_deficit']
acs_cols_summary = ['poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
                    'rent_burden_50pct_plus', 'unemployment_rate_pct']
indicator_names = ['Transit Dep.', 'Temporal Mis.', 'Structural Gap', 'Time Tax',
                   'Service Cov.', 'Econ. Vuln.', 'Multimodal Def.']
acs_names = ['Poverty %', 'No Vehicle %', 'SNAP %', 'Rent Burden %', 'Unemployment %']

s2_headers = ['Tier', 'Tracts', '% of Total'] + \
             [f'{n} (mean)' for n in indicator_names] + \
             ['Priority Score (mean)', 'Priority Score (median)'] + \
             [f'{n} (mean)' for n in acs_names]
ws2.append(s2_headers)
style_header(ws2, 1, len(s2_headers))

for tier in ['Critical', 'High', 'Moderate', 'Low']:
    tier_df = master[master['equity_tier'] == tier]
    n = len(tier_df)
    pct = n / len(master) * 100
    row_vals = [tier, n, round(pct, 1)]
    for col in indicator_cols:
        row_vals.append(round(tier_df[col].mean(), 3))
    row_vals.append(round(tier_df['equity_priority_score'].mean(), 3))
    row_vals.append(round(tier_df['equity_priority_score'].median(), 3))
    for col in acs_cols_summary:
        row_vals.append(round(tier_df[col].mean(), 1))
    ws2.append(row_vals)

# Total row
total_row = ['ALL', len(master), 100.0]
for col in indicator_cols:
    total_row.append(round(master[col].mean(), 3))
total_row.append(round(master['equity_priority_score'].mean(), 3))
total_row.append(round(master['equity_priority_score'].median(), 3))
for col in acs_cols_summary:
    total_row.append(round(master[col].mean(), 1))
ws2.append(total_row)

for r in range(2, ws2.max_row + 1):
    tier_val = ws2.cell(row=r, column=1).value
    for c in range(1, len(s2_headers) + 1):
        cell = ws2.cell(row=r, column=c)
        cell.font = DATA_FONT
        cell.alignment = Alignment(horizontal='center')
        cell.border = THIN_BORDER
    if tier_val in TIER_COLORS:
        ws2.cell(row=r, column=1).fill = TIER_COLORS[tier_val]
    if tier_val == 'ALL':
        for c in range(1, len(s2_headers) + 1):
            ws2.cell(row=r, column=c).font = Font(name='Arial', bold=True, size=10)

ws2.freeze_panes = 'B2'
auto_width(ws2)

# ============================================================
# SHEET 3: Top 20 Priority Tracts
# ============================================================
ws3 = wb.create_sheet('Top 20 Priority Tracts')

top20 = master.nlargest(20, 'equity_priority_score')

s3_cols = ['tract_geoid',
           'ind_1_transit_dependency', 'ind_2_temporal_mismatch', 'ind_3_structural_gap',
           'ind_4_time_tax', 'ind_5_service_coverage', 'ind_6_economic_vulnerability',
           'ind_7_multimodal_deficit',
           'equity_priority_score', 'equity_tier',
           'poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
           'rent_burden_50pct_plus', 'unemployment_rate_pct',
           'stop_count', 'route_count', 'daily_trip_count',
           'transit_jobs_30_mean', 'auto_jobs_30_mean',
           'time_tax_minutes_mean', 'transit_desert']
s3_headers = ['Census Tract',
              'Transit Dep.', 'Temporal Mis.', 'Structural Gap',
              'Time Tax', 'Service Cov.', 'Econ. Vuln.', 'Multimodal Def.',
              'Priority Score', 'Tier',
              'Poverty %', 'No Vehicle %', 'SNAP %',
              'Rent Burden %', 'Unemployment %',
              'Stops', 'Routes', 'Daily Trips',
              'Transit Jobs (30m)', 'Auto Jobs (30m)',
              'Time Tax (min)', 'Transit Desert']
# Only include columns that exist
valid_idx = [i for i, c in enumerate(s3_cols) if c in top20.columns]
s3_cols = [s3_cols[i] for i in valid_idx]
s3_headers = [s3_headers[i] for i in valid_idx]

ws3.append(s3_headers)
style_header(ws3, 1, len(s3_headers))

for _, row in top20.iterrows():
    vals = []
    for c in s3_cols:
        v = row[c]
        if isinstance(v, float) and c not in ['poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
                                                'rent_burden_50pct_plus', 'unemployment_rate_pct',
                                                'time_tax_minutes_mean']:
            vals.append(round(v, 3) if abs(v) < 10 else round(v, 0))
        else:
            vals.append(v)
    ws3.append(vals)

for r in range(2, ws3.max_row + 1):
    for c in range(1, len(s3_headers) + 1):
        cell = ws3.cell(row=r, column=c)
        cell.font = DATA_FONT
        cell.alignment = Alignment(horizontal='center')
        cell.border = THIN_BORDER

ws3.freeze_panes = 'B2'
auto_width(ws3)

# ============================================================
# SHEET 4: Methodology Reference
# ============================================================
ws4 = wb.create_sheet('Methodology Reference')

ws4.append(['Indicator', 'What It Measures', 'Key Inputs', 'Composite Role'])
style_header(ws4, 1, 4)

methodology = [
    ['1. Transit Dependency',
     'How reliant a community is on transit due to economic need + limited alternatives',
     'poverty_rate, no_vehicle_rate, transit vs auto job access ratio',
     'Need component (weighted 0.5)'],
    ['2. Temporal Mismatch',
     'Gap between when transit runs and when low-income/shift workers need it',
     'peak vs off-peak trip ratio, shift_worker_proxy, evening/late service levels',
     'Need component (weighted 0.5)'],
    ['3. Structural Gap',
     'How much worse transit job access is compared to auto in the same tract',
     'transit_jobs_30 / auto_jobs_30 ratio (inverted: higher = bigger gap)',
     'Access Deficit component (weighted ~0.33)'],
    ['4. Time Tax',
     'Extra minutes needed on transit to reach viable employment (10K+ jobs)',
     'Block-level transit time series → minutes to reach 10K jobs, aggregated to tract mean',
     'Access Deficit component (weighted ~0.33)'],
    ['5. Service Coverage',
     'Quality and density of transit service available in the tract',
     'stop_count, route_count, daily_trips, wheelchair_pct (weighted composite)',
     'Access Deficit component (weighted ~0.33)'],
    ['6. Economic Vulnerability',
     'Concentration of economic hardship indicators',
     'poverty 30%, SNAP 25%, rent_burden 25%, unemployment 20% (weighted)',
     'Need component (weighted 0.5)'],
    ['7. Multimodal Deficit',
     'How limited non-auto transportation options are overall',
     'transit 50%, bike (LTS1) 30%, walk (proxy) 20% — job access by mode',
     'Access Deficit component (weighted ~0.33)'],
    ['Composite: Need',
     'Combined demand-side pressure (who needs transit most)',
     'mean(Transit Dependency, Temporal Mismatch, Economic Vulnerability)',
     'Multiplied with Access Deficit'],
    ['Composite: Access Deficit',
     'Combined supply-side gap (where transit falls shortest)',
     'mean(Structural Gap, Time Tax, Service Coverage, Multimodal Deficit)',
     'Multiplied with Need'],
    ['Equity Priority Score',
     'Final priority = Need × Access Deficit (multiplicative)',
     'Concentrates score where BOTH need is high AND access is poor',
     'Percentile-ranked → 4 tiers: Critical (≥90th), High (≥70th), Moderate (≥40th), Low (<40th)'],
]

for row in methodology:
    ws4.append(row)

for r in range(2, ws4.max_row + 1):
    for c in range(1, 5):
        cell = ws4.cell(row=r, column=c)
        cell.font = DATA_FONT
        cell.alignment = Alignment(wrap_text=True, vertical='top')
        cell.border = THIN_BORDER
    ws4.cell(row=r, column=1).font = Font(name='Arial', bold=True, size=10)

ws4.column_dimensions['A'].width = 28
ws4.column_dimensions['B'].width = 50
ws4.column_dimensions['C'].width = 55
ws4.column_dimensions['D'].width = 35

ws4.freeze_panes = 'A2'

# --- Save ---
xlsx_path = 'Sprint2_Equity_Results_Overview_v3.xlsx'
wb.save(xlsx_path)
print(f"Results workbook saved: {xlsx_path}")
print(f"  Sheet 1: Equity Scores by Tract — {len(master)} tracts, sorted Critical→Low")
print(f"  Sheet 2: Summary by Tier — 4 tiers + ALL total row")
print(f"  Sheet 3: Top 20 Priority Tracts — full profile with ACS + GTFS")
print(f"  Sheet 4: Methodology Reference — 7 indicators + composite explanation")


Results workbook saved: Sprint2_Equity_Results_Overview_v3.xlsx
  Sheet 1: Equity Scores by Tract — 512 tracts, sorted Critical→Low
  Sheet 2: Summary by Tier — 4 tiers + ALL total row
  Sheet 3: Top 20 Priority Tracts — full profile with ACS + GTFS
  Sheet 4: Methodology Reference — 7 indicators + composite explanation


## 7. Final Notes (Changelog)

### v2 → v3 Changes
| Change | Rationale |
|--------|-----------|
| Block→Tract unit of analysis | ACS is tract-level; block granularity was illusory for demographic inputs |
| Block accessibility aggregated via mean | Preserves real within-tract variation in summary stats |
| Time Tax computed at block level, then averaged | Preserves granular transit time curves before aggregation |
| GTFS summed to tract level | Count measures (stops, trips) aggregate naturally |
| ~522 tracts instead of 36,507 blocks | Standard planning unit; every input at native resolution |
| Census tract crosswalk (2010→2020) | GeoPackage uses 2010 block geography; ACS uses 2020 tracts. Crosswalk translates IDs to achieve near-100% ACS match |

### Retained from v2
| Design Decision | Rationale |
|----------------|-----------|
| Walking 20min = to job, 5-10min = to transit stop | Walking 20min to a transit stop defeats the purpose of transit |
| Shift-worker inference via income proxy | State-level 29% shift rate correlates with low income |
| Multiplicative composite (Need × Deficit) | Concentrates priority where both conditions exist |
| LTS1 only for bike access | 32.8x gap between LTS4 and LTS1 |
| Time Tax → viable job threshold (10K) | Avoids comparing to auto (already captured in Structural Gap) |
| Service Coverage two-tier | Recognizes transit proximity for blocks without stops |
| Temporal Mismatch three-tier | Handles GTFS-served, transit-nearby, and no-transit tracts |

### Data Limitations
- GeoPackage reflects **peak-hour only** (7-9 AM) — off-peak accessibility unknown from this source
- Walk proxy uses bike LTS1 at 10min — slightly overestimates walk range
- Within-tract variation is captured as std columns but indicators use tract means
- GTFS spatial join may miss stops at block boundaries

### Next Steps
- Map indicators geographically (choropleth maps)
- Cross-reference with known underserved neighborhoods
- Sensitivity analysis on weights and thresholds
- Feed equity priority scores into route optimization model
